 Tokenization Project

This project explores different tokenization methods on *Shakespeare.txt*.
 Goals
- Compare Whitespace, *Regex**, Character, and BPE-lite tokenizers
- Compute statistics (vocab size, tokens per 1k chars, average token length, apostrophe handling)
- Visualize token distributions (Zipf plots)
- Examine tokenization of tricky Shakespearean contractions (*’tis, o’er, ne’er*)

In [14]:
from pathlib import Path
import re
from collections import Counter
import math
import matplotlib.pyplot as plt
from collections import Counter


DATA_PATH = Path("shakespeare.txt")
print("Data path:", DATA_PATH.resolve())
if not DATA_PATH.exists():
    raise FileNotFoundError("Place shakespeare.txt next to this script.")


Data path: C:\Users\DSU Student\Desktop\702\tokeniaztion\Shakespeare.txt



removing header/footer boilerplate.  

In [15]:
def load_clean_text(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="ignore")
    start_markers = [
        "*** START OF THIS PROJECT GUTENBERG EBOOK",
        "*** START OF THE PROJECT GUTENBERG EBOOK",
        "***START OF THE PROJECT GUTENBERG EBOOK",
    ]
    end_markers = [
        "*** END OF THIS PROJECT GUTENBERG EBOOK",
        "*** END OF THE PROJECT GUTENBERG EBOOK",
        "***END OF THE PROJECT GUTENBERG EBOOK",
    ]
    s_idx, e_idx = 0, len(text)
    for m in start_markers:
        i = text.find(m)
        if i != -1:
            j = text.find("\n", i)
            s_idx = j if j != -1 else i
            break
    for m in end_markers:
        i = text.find(m)
        if i != -1:
            e_idx = i
            break
    return text[s_idx:e_idx].strip()

text = load_clean_text(DATA_PATH)
print("Characters:", len(text))
print("Sample:\n", text[:500])


Characters: 5359343
Sample:
 ﻿The Complete Works of William Shakespeare

by William Shakespeare




                    Contents

    THE SONNETS
    ALL’S WELL THAT ENDS WELL
    THE TRAGEDY OF ANTONY AND CLEOPATRA
    AS YOU LIKE IT
    THE COMEDY OF ERRORS
    THE TRAGEDY OF CORIOLANUS
    CYMBELINE
    THE TRAGEDY OF HAMLET, PRINCE OF DENMARK
    THE FIRST PART OF KING HENRY THE FOURTH
    THE SECOND PART OF KING HENRY THE FOURTH
    THE LIFE OF KING HENRY THE FIFTH
    THE FIRST PART OF HENRY THE SIXTH
    THE SECOND P


 Tokenizers

implementing four tokenization strategies:
1. Whitespace – split on spaces
2. Regex – keeps words with apostrophes (’tis, o’er)
3. Character – each character is a token
4. BPE-lite – tiny byte-pair encoding (subword units), trained on a slice

In [16]:
# 1) Whitespace
def tok_whitespace(s: str):
    return s.split()

# 2) Regex word tokenizer 
WORD_APOS = r"[A-Za-z]+(?:['\u2019][A-Za-z]+)*"
TOKEN_RE  = re.compile(WORD_APOS)
def tok_regex(s: str):
    return TOKEN_RE.findall(s)

# 3) Character
def tok_char(s: str):
    return list(s)

# 4) Tiny educational BPE-like
def _to_char_words(words):
    return [list(w) + ["</w>"] for w in words]

def _count_pairs(token_words):
    pairs = Counter()
    for w in token_words:
        for a,b in zip(w, w[1:]):
            pairs[(a,b)] += 1
    return pairs

def _merge_pair(token_words, pair):
    a,b = pair
    merged = a+b
    new_vocab = []
    for w in token_words:
        i, new_w = 0, []
        while i < len(w):
            if i < len(w)-1 and w[i]==a and w[i+1]==b:
                new_w.append(merged); i += 2
            else:
                new_w.append(w[i]); i += 1
        new_vocab.append(new_w)
    return new_vocab

def train_bpe_vocab(words, merges=200):
    vocab = _to_char_words(words)
    merges_done = []
    for _ in range(merges):
        pairs = _count_pairs(vocab)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        vocab = _merge_pair(vocab, best)
        merges_done.append("".join(best))
    return merges_done

def apply_bpe(word, merges_done):
    
    pieces = list(word) + ["</w>"]
    merged = True
    while merged:
        merged = False
        i = 0
        while i < len(pieces)-1:
            candidate = pieces[i] + pieces[i+1]
            if candidate in merges_done:
                pieces[i:i+2] = [candidate]
                merged = True
            else:
                i += 1
    return [p for p in pieces if p != "</w>"]

def tok_bpe(s: str, merges_done):
    words = TOKEN_RE.findall(s.lower())
    out = []
    for w in words:
        out.extend(apply_bpe(w, merges_done))
    return out


Train BPE & Build Tokenizers

In [17]:
tokenizers = {
    "whitespace": tok_whitespace,
    "regex": tok_regex,
    "char": tok_char,
}

# Train BPE on a subset of words (for speed, 20k words)
sample_words = tok_regex(text.lower())[:20000]
merges_done = train_bpe_vocab(sample_words, merges=300)
tokenizers["bpe-lite"] = lambda s: tok_bpe(s, merges_done)

print("Tokenizers ready:", list(tokenizers.keys()))
print("BPE merges learned:", len(merges_done))


Tokenizers ready: ['whitespace', 'regex', 'char', 'bpe-lite']
BPE merges learned: 300




 Evaluation Metrics

For each tokenizer we compute:
Total tokens
Vocabulary size
Tokens per 1,000 characters
Average token length
Share of tokens with apostrophes
Top-20 most frequent tokens


In [18]:
def eval_tokenizer(name, fn, s: str, max_len=None):
    chunk = s if not max_len else s[:max_len]
    toks = fn(chunk)
    vocab = set(toks)
    n_chars = len(chunk)
    n_toks = len(toks)
    voc_size = len(vocab)
    avg_tok_len = (sum(len(t) for t in toks)/max(1,n_toks)) if name!="char" else 1.0
    apos_share = sum(1 for t in toks if ("'" in t or "\u2019" in t)) / max(1,n_toks)
    tpk = (n_toks / max(1,n_chars)) * 1000.0  # tokens per 1000 chars
    return {
        "name": name,
        "chars": n_chars,
        "tokens": n_toks,
        "tokens_per_1k_chars": tpk,
        "vocab_size": voc_size,
        "avg_token_len": avg_tok_len,
        "apostrophe_token_share": apos_share,
        "top20": Counter(toks).most_common(20),
    }

def pretty_print_eval(stats):
    print(f"\n== {stats['name']} ==")
    print(f"chars={stats['chars']:,} | tokens={stats['tokens']:,} | vocab={stats['vocab_size']:,}")
    print(f"tokens/1k chars={stats['tokens_per_1k_chars']:.1f} | avg_token_len={stats['avg_token_len']:.2f} | apostrophe_share={stats['apostrophe_token_share']:.3f}")
    print("Top 20:", stats["top20"])


Run Evaluations

In [19]:
SLICE = 400_000  # adjust for speed; increase for full run

results = []
for name, fn in tokenizers.items():
    stats = eval_tokenizer(name, fn, text, max_len=SLICE)
    results.append(stats)
    pretty_print_eval(stats)



== whitespace ==
chars=400,000 | tokens=71,712 | vocab=13,217
tokens/1k chars=179.3 | avg_token_len=4.47 | apostrophe_share=0.028
Top 20: [('the', 1686), ('I', 1588), ('and', 1290), ('to', 1272), ('of', 1252), ('my', 977), ('in', 871), ('a', 846), ('that', 687), ('you', 655), ('is', 644), ('not', 568), ('And', 528), ('his', 521), ('with', 493), ('have', 451), ('be', 439), ('me', 429), ('thou', 425), ('for', 424)]

== regex ==
chars=400,000 | tokens=71,873 | vocab=8,457
tokens/1k chars=179.7 | avg_token_len=4.21 | apostrophe_share=0.021
Top 20: [('the', 1696), ('I', 1658), ('and', 1307), ('to', 1295), ('of', 1273), ('my', 980), ('in', 890), ('you', 878), ('a', 858), ('that', 718), ('is', 676), ('me', 651), ('not', 643), ('it', 558), ('And', 536), ('his', 524), ('with', 502), ('be', 489), ('have', 472), ('thou', 450)]

== char ==
chars=400,000 | tokens=400,000 | vocab=84
tokens/1k chars=1000.0 | avg_token_len=1.00 | apostrophe_share=0.005
Top 20: [(' ', 64612), ('e', 34436), ('t', 24087

 Tricky example tokenizations to check the tokenizers
A short set of Shakespearean lines used to demonstrate how each tokenizer handles contractions, apostrophes, hyphens and archaic forms. Prints token lists for inspection.

In [21]:
examples = [
    "’Tis not alone my inky cloak, good mother.",
    "O’er hill, o’er dale, thorough bush, thorough brier.",
    "Ne’er was seen so black a day as this.",
    "Love’s labour’s lost; all’s well that ends well.",
    "Self-love, my liege, is not so vile a sin as self-neglecting.",
    "Thou art as fat as butter.",
]
for s in examples:
    print("\nLINE:", s)
    for name, fn in tokenizers.items():
        toks = fn(s)
        print(f"{name:>10}:", toks[:40])



LINE: ’Tis not alone my inky cloak, good mother.
whitespace: ['’Tis', 'not', 'alone', 'my', 'inky', 'cloak,', 'good', 'mother.']
     regex: ['Tis', 'not', 'alone', 'my', 'inky', 'cloak', 'good', 'mother']
      char: ['’', 'T', 'i', 's', ' ', 'n', 'o', 't', ' ', 'a', 'l', 'o', 'n', 'e', ' ', 'm', 'y', ' ', 'i', 'n', 'k', 'y', ' ', 'c', 'l', 'o', 'a', 'k', ',', ' ', 'g', 'o', 'o', 'd', ' ', 'm', 'o', 't', 'h', 'e']
  bpe-lite: ['ti', 's</w>', 'not</w>', 'al', 'one</w>', 'my</w>', 'in', 'k', 'y</w>', 'c', 'lo', 'a', 'k</w>', 'go', 'o', 'd</w>', 'mo', 'ther</w>']

LINE: O’er hill, o’er dale, thorough bush, thorough brier.
whitespace: ['O’er', 'hill,', 'o’er', 'dale,', 'thorough', 'bush,', 'thorough', 'brier.']
     regex: ['O’er', 'hill', 'o’er', 'dale', 'thorough', 'bush', 'thorough', 'brier']
      char: ['O', '’', 'e', 'r', ' ', 'h', 'i', 'l', 'l', ',', ' ', 'o', '’', 'e', 'r', ' ', 'd', 'a', 'l', 'e', ',', ' ', 't', 'h', 'o', 'r', 'o', 'u', 'g', 'h', ' ', 'b', 'u', 's', 'h', ',', ' 

Summary table
Formats and prints a compact summary table (tokenizer, chars, tokens, tokens/1k chars, vocab, avg token length, apostrophe share) for all evaluated tokenizers.

In [24]:
def row(st):
    return [
        st["name"],
        f"{st['chars']:,}",
        f"{st['tokens']:,}",
        f"{st['tokens_per_1k_chars']:.1f}",
        f"{st['vocab_size']:,}",
        f"{st['avg_token_len']:.2f}",
        f"{st['apostrophe_token_share']:.3f}",
    ]

headers = ["tokenizer","chars","tokens","tok/1k chars","vocab","avg tok len","apos share"]
print("\n" + " | ".join(headers))
print("-"*80)
for st in results:
    print(" | ".join(row(st)))



tokenizer | chars | tokens | tok/1k chars | vocab | avg tok len | apos share
--------------------------------------------------------------------------------
whitespace | 400,000 | 71,712 | 179.3 | 13,217 | 4.47 | 0.028
regex | 400,000 | 71,873 | 179.7 | 8,457 | 4.21 | 0.021
char | 400,000 | 400,000 | 1000.0 | 84 | 1.00 | 0.005
bpe-lite | 400,000 | 153,987 | 385.0 | 321 | 3.77 | 0.010


Subword / BPE Details

To better understand how Byte-Pair Encoding (BPE) works, we look at:
1. The first 20 merges learned** (common character pairs that get fused early, like `th`, `he`, `in`).
2. How the tokenization of the word "therefore" evolves** as we increase the number of merges:
    At 0 merges → only characters.
    At 50 merges → some common bigrams like `th` appear.
    At 300 merges → full subwords or whole words may appear.

This demonstrates how BPE gradually builds a compact subword vocabulary.


In [27]:
# Print the first 20 BPE merges learned
print("First 20 BPE merges learned:")
for i, merge in enumerate(merges_done[:20], 1):
    print(f"{i:2d}. {merge}")


First 20 BPE merges learned:
 1. e</w>
 2. th
 3. t</w>
 4. s</w>
 5. d</w>
 6. y</w>
 7. in
 8. r</w>
 9. ou
10. an
11. o</w>
12. en
13. ea
14. l</w>
15. f</w>
16. on
17. er
18. and</w>
19. the</w>
20. th</w>


In [28]:
test_word = "therefore"

# Train vocab with smaller number of merges for comparison
merges_50  = train_bpe_vocab(sample_words, merges=50)
merges_300 = train_bpe_vocab(sample_words, merges=300)

print(f"\nTokenizing '{test_word}' at different merge levels:")

print("0 merges (characters):", list(test_word))
print("50 merges:", apply_bpe(test_word, merges_50))
print("300 merges:", apply_bpe(test_word, merges_300))



Tokenizing 'therefore' at different merge levels:
0 merges (characters): ['t', 'h', 'e', 'r', 'e', 'f', 'o', 'r', 'e']
50 merges: ['the', 'r', 'e', 'f', 'or', 'e</w>']
300 merges: ['the', 're', 'fore</w>']


At 0 merges, “therefore” is split entirely into characters. After 50 merges, frequent substrings such as “the” and “or” appear, showing how BPE first compresses the most common patterns. By 300 merges, larger morphemes like “fore” are learned, reducing the word to only three tokens. This illustrates how BPE gradually builds subword units that balance between efficiency and coverage.